# Errors and limits

Every rejected job returns the same envelope: an HTTP status, a typed error and machine-readable details. Clients can
branch on the type instead of parsing messages. This notebook triggers each common error on purpose.

In [1]:
from qputwin_demo import *

connect()

QPUTwin at http://localhost:8000: ready, 7 twins reachable


{'status': 'ready',
 'instance': 'c88add07efb8:7:9e791f',
 'job_store': 'ok',
 'backends_reachable': ['quantinuum_helios',
  'ionq_forte',
  'rigetti_ankaa_3',
  'iqm_emerald',
  'ibm_heron',
  'ibm_nighthawk',
  'google_willow']}

## 1. A catalogue of errors

One request for each common mistake, and what comes back.

In [2]:
cases = {
    "unknown backend": ("no_such_device", bell(), {}),
    "too many shots": ("quantinuum_helios", bell(), {"shots": 10_000_000}),
    "malformed program": ("quantinuum_helios", HEADER + "qreg q[1];\nh q[0]\n", {}),
    "uncoupled qubits, no routing": ("iqm_emerald", program(3, ["cx q[0],q[2]"]), {}),
    "too wide for memory": ("quantinuum_helios", ghz(40), {}),
    "noise with the statevector engine": ("quantinuum_helios", bell(), {"noise": True, "engine": "statevector"}),
    "strict mode with a non-native gate": ("iqm_emerald", bell(), {"compilation": "strict"}),
}
rows = []
for name, (b, prog, kw) in cases.items():
    r = run(b, prog, **kw)
    e = r.get("error", {})
    rows.append({"case": name, "HTTP": r["http_status"], "type": e.get("type"), "message": (e.get("message") or "")[:90]})
pd.set_option("display.max_colwidth", 90)
pd.DataFrame(rows).set_index("case")

,HTTP,type,message
case,,,
unknown backend,404,UnknownBackendError,unknown backend 'no_such_device'
too many shots,400,InvalidRequestError,request body failed validation
malformed program,400,QASMSyntaxError,"expected ';' after gate call, found end of input (line 5, col 1)"
"uncoupled qubits, no routing",422,TopologicalCompilationError,"'cx q[0],q[2];' (line 5) requires a two-qubit interaction between physical qubits 0 an..."
too wide for memory,422,ResourceLimitError,a dense 40-qubit complex128 statevector run needs ~24.0 TiB but only 8.5 GiB is availa...
noise with the statevector engine,400,InvalidRequestError,"noisy execution needs the density-matrix or trajectory engine; use engine 'auto', 'den..."
strict mode with a non-native gate,422,NativeGateError,"'h' (line 5) lowers to non-native operation 'U'; iqm_emerald natives are ['cz', 'prx',..."


## 2. Details are machine-readable

The `details` object carries the facts behind each error, for example the uncoupled pair and the device, or the memory a job would need.

In [3]:
for name in ("uncoupled qubits, no routing", "too wide for memory"):
    b, prog, kw = cases[name]
    print(name, "->", run(b, prog, **kw)["error"]["details"])

uncoupled qubits, no routing -> {'device': 'iqm_emerald', 'gate': 'cx', 'primitive': 'CX', 'qubits': [0, 2], 'line': 5, 'statement': 'cx q[0],q[2];', 'neighbors': {'0': [1, 9], '2': [1, 3, 11]}, 'adjacency_entry': 0}
too wide for memory -> {'simulated_qubits': 40, 'required_bytes': 26388279066624, 'available_bytes': 9144238080, 'max_feasible_qubits': 28, 'precision': 'complex128', 'device': 'quantinuum_helios', 'engine': 'statevector'}


## 3. Validation errors list every problem

A request with several invalid fields is rejected once, with one entry per problem.

In [4]:
import httpx
bad = {"backend": "quantinuum_helios", "qasm": bell(), "shots": 0, "priority": 42, "options": {"precision": "complex256"}}
r = httpx.post(f"{API_URL}/v1/jobs", json=bad)
print("HTTP", r.status_code, r.json()["error"]["type"])
for e in r.json()["error"]["details"]["errors"]:
    print(" ", e["loc"], "->", e["msg"])

HTTP 400 InvalidRequestError
  ['body', 'shots'] -> Input should be greater than or equal to 1
  ['body', 'options', 'precision'] -> Input should be 'complex128' or 'complex64'
  ['body', 'priority'] -> Input should be less than or equal to 9


## 4. Cancelling: only queued jobs

A queued asynchronous job can be cancelled; a finished one cannot, and says why.

In [5]:
slow = [submit("ionq_forte", scrambler(14, 10), shots=4000, noise=True)["job_id"] for _ in range(4)]
last = slow[-1]
print("cancel a queued job:", cancel(last).get("status") or cancel(last))
first = slow[0]
while job(first, wait=20)["status"] != "COMPLETED":
    pass
print("cancel a finished job:", cancel(first).get("error", {}).get("type"))
for jid in slow[1:-1]:
    cancel(jid)

cancel a queued job: CANCELLED


cancel a finished job: JobStateError


## 5. The service's own limits

`GET /v1/backends/{id}` reports each twin's limits under `emulator`, including the memory currently available to its worker, so a client can check a job before sending it.

In [6]:
emu = device("iqm_emerald")["emulator"]
keys = ["max_sim_qubits", "max_shots", "max_statevector_qubits", "max_density_matrix_qubits", "max_trajectories",
        "density_auto_max_qubits", "available_memory_bytes", "precision", "engines"]
pd.Series({k: emu[k] for k in keys}, name="iqm_emerald")

max_sim_qubits                                                      54
max_shots                                                      1000000
max_statevector_qubits                                              16
max_density_matrix_qubits                                            8
max_trajectories                                                  2000
density_auto_max_qubits                                             12
available_memory_bytes                                     10273574912
precision                                                   complex128
engines                      [statevector, density_matrix, trajectory]
Name: iqm_emerald, dtype: object

## Takeaways

- Every rejection is typed and carries structured details.
- Invalid jobs are refused before they reach a worker, so they cost milliseconds.